In [2]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 31.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
import numpy as np
from scipy import sparse
from sklearn.model_selection import train_test_split

X_train_district_processed = sparse.load_npz(
    "week6_X_train_district_processed.npz"
)

X_test_district_processed = sparse.load_npz(
    "week6_X_test_district_processed.npz"
)

y_train_week6 = np.load("week6_y_train.npy")
y_test_week6 = np.load("week6_y_test.npy")

print("Training shape:", X_train_district_processed.shape)
print("Test shape:", X_test_district_processed.shape)

Training shape: (128568, 4646)
Test shape: (11914, 4646)


In [6]:
X_train_xgb, X_validation_xgb, y_train_xgb, y_validation_xgb = (
    train_test_split(
        X_train_district_processed,
        y_train_week6,
        test_size=0.2,
        random_state=42
    )
)

print("Training shape:", X_train_xgb.shape)
print("Validation shape:", X_validation_xgb.shape)
print("Test shape:", X_test_district_processed.shape)

Training shape: (102854, 4646)
Validation shape: (25714, 4646)
Test shape: (11914, 4646)


In [7]:
xgb_results = []

for max_depth in [4, 6, 8]:
    for learning_rate in [0.03, 0.05, 0.1]:
        for n_estimators in [100, 200, 300]:

            model = XGBRegressor(
                max_depth=max_depth,
                learning_rate=learning_rate,
                n_estimators=n_estimators,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1
            )

            model.fit(X_train_xgb, y_train_xgb)
            validation_predictions = model.predict(X_validation_xgb)

            xgb_results.append({
                "max_depth": max_depth,
                "learning_rate": learning_rate,
                "n_estimators": n_estimators,
                "Validation_R2": r2_score(
                    y_validation_xgb,
                    validation_predictions
                ),
                "Validation_MAE": mean_absolute_error(
                    y_validation_xgb,
                    validation_predictions
                ),
                "Validation_RMSE": np.sqrt(
                    mean_squared_error(
                        y_validation_xgb,
                        validation_predictions
                    )
                )
            })

xgb_tuning_results = pd.DataFrame(xgb_results)

xgb_tuning_results.sort_values(
    "Validation_R2",
    ascending=False
).head(10)

,max_depth,learning_rate,n_estimators,Validation_R2,Validation_MAE,Validation_RMSE
26,8,0.10,300,0.887418,162104.635023,289327.789872
25,8,0.10,200,0.881323,168842.381774,297056.616310
23,8,0.05,300,0.879872,170977.343461,298867.095136
17,6,0.10,300,0.879515,173460.543079,299309.912715
22,8,0.05,200,0.873136,177422.177140,307131.787142
20,8,0.03,300,0.870843,179324.814134,309894.293778
16,6,0.10,200,0.870485,182756.135280,310324.109665
24,8,0.10,100,0.869843,180081.251445,311092.520488
19,8,0.03,200,0.863039,186182.207032,319120.323829
14,6,0.05,300,0.860455,190671.731602,322115.686372


In [8]:
best_parameters = (
    xgb_tuning_results
    .sort_values("Validation_R2", ascending=False)
    .iloc[0]
)

best_parameters

max_depth               8.000000
learning_rate           0.100000
n_estimators          300.000000
Validation_R2           0.887418
Validation_MAE     162104.635023
Validation_RMSE    289327.789872
Name: 26, dtype: float64

In [9]:
best_xgb_model = XGBRegressor(
    max_depth=int(best_parameters["max_depth"]),
    learning_rate=float(best_parameters["learning_rate"]),
    n_estimators=int(best_parameters["n_estimators"]),
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

best_xgb_model.fit(
    X_train_district_processed,
    y_train_week6
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [10]:
xgb_test_predictions = best_xgb_model.predict(
    X_test_district_processed
)

xgb_test_r2 = r2_score(
    y_test_week6,
    xgb_test_predictions
)

xgb_test_mae = mean_absolute_error(
    y_test_week6,
    xgb_test_predictions
)

xgb_test_rmse = np.sqrt(
    mean_squared_error(
        y_test_week6,
        xgb_test_predictions
    )
)

print("XGBoost Test Results")
print("R²:", xgb_test_r2)
print("MAE:", xgb_test_mae)
print("RMSE:", xgb_test_rmse)

XGBoost Test Results
R²: 0.8894820885810153
MAE: 169190.1746146865
RMSE: 301126.91809754726


In [11]:
week7_comparison = pd.DataFrame([
    {
        "Model": "Random Forest",
        "R2": 0.865646,
        "MAE": 173669.729391,
        "RMSE": 332015.107951
    },
    {
        "Model": "XGBoost",
        "R2": xgb_test_r2,
        "MAE": xgb_test_mae,
        "RMSE": xgb_test_rmse
    }
])

week7_comparison

,Model,R2,MAE,RMSE
0,Random Forest,0.865646,173669.729391,332015.107951
1,XGBoost,0.889482,169190.174615,301126.918098
